In [ ]:
from src.data.data import *
from src.embedor import *
from src.plotting import *
import matplotlib
import seaborn as sns
import umap
import numpy as np
from sklearn.manifold import TSNE, Isomap, SpectralEmbedding
import phate

%load_ext autoreload

In [310]:

def get_low_energy_graph(embedor_object: EmbedOR, edge_pctile=33):
    """
    Returns a graph with edges that are below the specified percentile of edge distances.
    """
    G_low_energy = embedor_object.G.copy()
    edge_dists = {
        (u, v): embedor_object.G[u][v]['energy'] for u, v in embedor_object.G.edges
    }
    edge_distances = np.array(list(edge_dists.values()))
    for idx, (u, v) in enumerate(embedor_object.G.edges):
        if edge_distances[idx] > np.percentile(edge_distances, edge_pctile):
            G_low_energy.remove_edge(u, v)
    return G_low_energy

def eval_low_energy_edges(embedor_object: EmbedOR, edge_pctile=33, clusters=None):
    # get number of cluster bridging edges in the entire graph
    G_full = embedor_object.G.copy()
    n_brige_full = 0
    for edge in G_full.edges:
        u, v = edge
        if clusters[u] != clusters[v]:
            n_brige_full += 1
    # check what percent of low energy edges bridge the clusters
    G_low_energy = get_low_energy_graph(embedor_object, edge_pctile=edge_pctile)
    n_bridge_low_energy = 0
    for edge in G_low_energy.edges:
        u, v = edge
        if clusters[u] != clusters[v]:
            n_bridge_low_energy += 1
    return n_bridge_low_energy, n_brige_full


exp_params = {
    'p': 3,
}

In [ ]:
def circles_metric_eval(n_iter=10):
    n_points = 5000
    # concentric circles
    noise = 0.1
    noise_thresh = None
    n_bridge_full_array = []
    n_bridge_low_energy_array = []
    for iter in range(n_iter):
        print(f"Iteration {iter+1}/{n_iter}")
        return_dict = concentric_circles(n_points=n_points, factor=0.4, noise=noise, noise_thresh=noise_thresh)
        embedor = EmbedOR(exp_params)
        embedding = embedor.fit_transform(return_dict['data'])
        n_bridge_low_energy, n_brige_full = eval_low_energy_edges(embedor, edge_pctile=33, clusters=return_dict['cluster'])
        n_bridge_low_energy_array.append(n_bridge_low_energy)
        n_bridge_full_array.append(n_brige_full)

    n_bridge_full_array = np.array(n_bridge_full_array)
    n_bridge_low_energy_array = np.array(n_bridge_low_energy_array)
    return n_bridge_low_energy_array, n_bridge_full_array

circles_n_bridge_low_energy, circles_n_bridge_full = circles_metric_eval(n_iter=10)
print(f'(mean, std) #  bridging edges in low energy graph: {np.mean(circles_n_bridge_low_energy):.2f} ± {np.std(circles_n_bridge_low_energy):.2f}')
print(f'(mean, std) #  bridging edges in full graph: {np.mean(circles_n_bridge_full):.2f} ± {np.std(circles_n_bridge_full):.2f}')

Iteration 1/10
Building nearest neighbor graph...
Computing distances...
Computing affinities...
Updating the graph attributes...
Running Stochastic Neighbor Embedding...
Iteration 2/10
Building nearest neighbor graph...
Computing distances...
Computing affinities...
Updating the graph attributes...
Running Stochastic Neighbor Embedding...
Iteration 3/10
Building nearest neighbor graph...
Computing distances...
Computing affinities...
Updating the graph attributes...
Running Stochastic Neighbor Embedding...
Iteration 4/10
Building nearest neighbor graph...
Computing distances...
Computing affinities...
Updating the graph attributes...
Running Stochastic Neighbor Embedding...
Iteration 5/10
Building nearest neighbor graph...
Computing distances...
Computing affinities...
Updating the graph attributes...
Running Stochastic Neighbor Embedding...
Iteration 6/10
Building nearest neighbor graph...
Computing distances...
Computing affinities...
Updating the graph attributes...
Running Stochas

In [312]:

def moons_metric_eval(n_iter=10):
    n_points = 3000
    noise = 0.125
    noise_thresh = None
    n_bridge_full_array = []
    n_bridge_low_energy_array = []
    for iter in range(n_iter):
        print(f"Iteration {iter+1}/{n_iter}")
        return_dict = moons(n_points=n_points, noise=noise, noise_thresh=noise_thresh)
        embedor = EmbedOR(exp_params)
        embedding = embedor.fit_transform(return_dict['data'])
        n_bridge_low_energy, n_brige_full = eval_low_energy_edges(embedor, edge_pctile=33, clusters=return_dict['cluster'])
        n_bridge_low_energy_array.append(n_bridge_low_energy)
        n_bridge_full_array.append(n_brige_full)
    n_bridge_full_array = np.array(n_bridge_full_array)
    n_bridge_low_energy_array = np.array(n_bridge_low_energy_array)
    return n_bridge_low_energy_array, n_bridge_full_array

moons_n_bridge_low_energy, moons_n_bridge_full = moons_metric_eval(n_iter=10)
print(f'(mean, std) #  bridging edges in low energy graph: {np.mean(moons_n_bridge_low_energy):.2f} ± {np.std(moons_n_bridge_low_energy):.2f}')
print(f'(mean, std) #  bridging edges in full graph: {np.mean(moons_n_bridge_full):.2f} ± {np.std(moons_n_bridge_full):.2f}')   


Iteration 1/10
Building nearest neighbor graph...
Computing distances...
Computing affinities...
Updating the graph attributes...
Running Stochastic Neighbor Embedding...
Iteration 2/10
Building nearest neighbor graph...
Computing distances...
Computing affinities...
Updating the graph attributes...
Running Stochastic Neighbor Embedding...
Iteration 3/10
Building nearest neighbor graph...
Computing distances...
Computing affinities...
Updating the graph attributes...
Running Stochastic Neighbor Embedding...
Iteration 4/10
Building nearest neighbor graph...
Computing distances...
Computing affinities...
Updating the graph attributes...
Running Stochastic Neighbor Embedding...
Iteration 5/10
Building nearest neighbor graph...
Computing distances...
Computing affinities...
Updating the graph attributes...
Running Stochastic Neighbor Embedding...
Iteration 6/10
Building nearest neighbor graph...
Computing distances...
Computing affinities...
Updating the graph attributes...
Running Stochas

In [294]:
noise = 0.5
n_points = 5000
noise_thresh = None
return_dict = torus(n_points=n_points, noise=noise, noise_thresh=noise_thresh, supersample=False, double=True)

In [313]:
def torus_metric_eval(n_iter=10):
    n_points = 5000
    noise = 0.5
    noise_thresh = None
    n_bridge_full_array = []
    n_bridge_low_energy_array = []
    for iter in range(n_iter):
        print(f"Iteration {iter+1}/{n_iter}")
        return_dict = torus(n_points=n_points, noise=noise, noise_thresh=noise_thresh, supersample=False, double=True)
        embedor = EmbedOR(exp_params)
        embedding = embedor.fit_transform(return_dict['data'])
        n_bridge_low_energy, n_brige_full = eval_low_energy_edges(embedor, edge_pctile=33, clusters=return_dict['cluster'])
        n_bridge_low_energy_array.append(n_bridge_low_energy)
        n_bridge_full_array.append(n_brige_full)
    n_bridge_full_array = np.array(n_bridge_full_array)
    n_bridge_low_energy_array = np.array(n_bridge_low_energy_array)
    return n_bridge_low_energy_array, n_bridge_full_array

torus_n_bridge_low_energy, torus_n_bridge_full = torus_metric_eval(n_iter=10)
print(f'(mean, std) #  bridging edges in low energy graph: {np.mean(torus_n_bridge_low_energy):.2f} ± {np.std(torus_n_bridge_low_energy):.2f}')
print(f'(mean, std) #  bridging edges in full graph: {np.mean(torus_n_bridge_full):.2f} ± {np.std(torus_n_bridge_full):.2f}')

Iteration 1/10
Building nearest neighbor graph...
Computing distances...
Computing affinities...
Updating the graph attributes...
Running Stochastic Neighbor Embedding...
Iteration 2/10
Building nearest neighbor graph...
Computing distances...
Computing affinities...
Updating the graph attributes...
Running Stochastic Neighbor Embedding...
Iteration 3/10
Building nearest neighbor graph...
Computing distances...
Computing affinities...
Updating the graph attributes...
Running Stochastic Neighbor Embedding...
Iteration 4/10
Building nearest neighbor graph...
Computing distances...
Computing affinities...
Updating the graph attributes...
Running Stochastic Neighbor Embedding...
Iteration 5/10
Building nearest neighbor graph...
Computing distances...
Computing affinities...
Updating the graph attributes...
Running Stochastic Neighbor Embedding...
Iteration 6/10
Building nearest neighbor graph...
Computing distances...
Computing affinities...
Updating the graph attributes...
Running Stochas

In [ ]:
# %autoreload 2
# edge_energies = {
#     (u, v): embedor.G[u][v]['energy'] for u, v in embedor.G.edges
# }
# edge_energies = np.array(list(edge_energies.values()))
# # plot_graph_2D(return_dict['data'], embedor.G, node_color=return_dict['cluster'], edge_width=0.3, node_size=0.1, edge_color=edge_energies)

# edge_distances = {
#     (u, v): embedor.G[u][v]['weight'] for u, v in embedor.G.edges
# }
# edge_distances = np.array(list(edge_distances.values()))

# cluster_bridging_edges = [1 if return_dict['cluster'][u] != return_dict['cluster'][v] else 0 for u, v in embedor.G.edges]
# cluster_bridging_edges = np.array(cluster_bridging_edges)
# plot_graph_2D(return_dict['data'], embedor.G, node_color=None, edge_width=0.3, node_size=0.1, edge_color=edge_energies)

In [298]:
# cluster_bridging_edge_distance_percentiles, cluster_bridging_edge_energy_percentiles = eval_metric(embedor, clusters=return_dict['cluster'])
# # plot histograms of the percentiles overlaid
# plt.figure(figsize=(10, 5))
# plt.hist(cluster_bridging_edge_distance_percentiles, bins=100, alpha=0.5, label='Cluster Bridging Edge Distance Percentiles', cumulative=True, density=True)
# plt.hist(cluster_bridging_edge_energy_percentiles, bins=100, alpha=0.5, label='Cluster Bridging Edge Energy Percentiles', cumulative=True, density=True)
# plt.xlabel('Percentile')
# plt.ylabel('Count')
# plt.title('Distribution of Cluster Bridging Edge Percentiles')
# plt.legend()

In [ ]:
# print(np.mean(cluster_bridging_edge_distance_percentiles), np.mean(cluster_bridging_edge_energy_percentiles)) 

In [299]:
# plot cdf of cluster bridging edge distance percentiles wrt apsp
n_points = 5000
# concentric circles
noise = 0.1
noise_thresh = None

return_dict = concentric_circles(n_points=n_points, factor=0.4, noise=noise, noise_thresh=noise_thresh)
embedor = EmbedOR(exp_params)
embedding = embedor.fit_transform(return_dict['data'])

Building nearest neighbor graph...
Computing distances...
Computing affinities...
Updating the graph attributes...
Running Stochastic Neighbor Embedding...


Edge 0-2500 has percentile 48.42
Edge 0-2501 has percentile 43.93
Edge 0-2502 has percentile 44.09
Edge 0-2503 has percentile 49.10
Edge 0-2504 has percentile 45.85
Edge 0-2505 has percentile 46.55
Edge 0-2506 has percentile 50.35
Edge 0-2507 has percentile 40.24
Edge 0-2508 has percentile 40.67
Edge 0-2509 has percentile 48.82
Edge 0-2510 has percentile 43.23
Edge 0-2511 has percentile 39.59
Edge 0-2512 has percentile 47.62
Edge 0-2513 has percentile 44.22
Edge 0-2514 has percentile 46.11
Edge 0-2515 has percentile 55.80
Edge 0-2516 has percentile 51.20
Edge 0-2517 has percentile 41.45
Edge 0-2518 has percentile 46.00
Edge 0-2519 has percentile 44.67
Edge 0-2520 has percentile 49.79
Edge 0-2521 has percentile 38.71
Edge 0-2522 has percentile 50.80
Edge 0-2523 has percentile 52.69
Edge 0-2524 has percentile 48.32
Edge 0-2525 has percentile 45.13
Edge 0-2526 has percentile 13.37
Edge 0-2527 has percentile 37.88
Edge 0-2528 has percentile 44.48
Edge 0-2529 has percentile 46.36
Edge 0-253

KeyboardInterrupt: 